## Mode Selection

In [ ]:
import os, sys
os.environ["PYTHONIOENCODING"] = "utf-8"
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"
os.environ["TORCH_COMPILE_DISABLE"] = "1"
os.environ["TORCHINDUCTOR_DISABLE"] = "1"
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="strict")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="strict")

# Mode A: Train LoRA from scratch on Kaggle GPU
TRAIN_ON_KAGGLE = 1

# Mode B: Use pre-trained LoRA weights from dataset and just package them
USE_PRETRAINED = 0

assert (TRAIN_ON_KAGGLE + USE_PRETRAINED) == 1, \
    "Set exactly one of TRAIN_ON_KAGGLE / USE_PRETRAINED to 1."

PRETRAINED_ADAPTER_DATASET_PATH = "/kaggle/input/datasets/konbu17/nemotron-sft-lora-cot-selection"
BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"

print({
    "TRAIN_ON_KAGGLE": TRAIN_ON_KAGGLE,
    "USE_PRETRAINED": USE_PRETRAINED,
    "PRETRAINED_ADAPTER_DATASET_PATH": PRETRAINED_ADAPTER_DATASET_PATH,
})


## Setup & Model Loading

In [ ]:
import os, glob, sys, subprocess, site

candidates = glob.glob("/kaggle/input/**/*triton*.whl", recursive=True)
print("Found Triton wheels:", candidates)

if not candidates:
    raise FileNotFoundError("No Triton wheel found under /kaggle/input")
wheel = candidates[0]

target = "/kaggle/working/pydeps"
os.makedirs(target, exist_ok=True)

subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--no-deps",
        "--target", target,
        "--upgrade",
        "--ignore-installed",
        wheel,
    ],
    check=True,
)

if target not in sys.path:
    sys.path.insert(0, target)

site.addsitedir(target)

print("Custom target added:", target)

import importlib.util
print("triton spec：", importlib.util.find_spec("triton"))


In [ ]:
if TRAIN_ON_KAGGLE:
    import sys, os, shutil, stat

    # Add utility script to Python path (provides helper binaries)
    sys.path.insert(0, '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script')

    # Copy ptxas-blackwell to /tmp with execute permissions
    ptxas_src = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell'
    ptxas_dst = '/tmp/ptxas-blackwell'
    if os.path.exists(ptxas_src) and not os.path.exists(ptxas_dst):
        shutil.copy2(ptxas_src, ptxas_dst)
        os.chmod(ptxas_dst, os.stat(ptxas_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

        src_bin = os.path.dirname(ptxas_src)
        dst_bin = '/tmp/triton_nvidia_bin'
        shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
        for f in os.listdir(dst_bin):
            fp = os.path.join(dst_bin, f)
            if os.path.isfile(fp):
                os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

        os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = ptxas_dst

        import triton.backends.nvidia as nv_backend
        nv_backend.__file__ = os.path.join(dst_bin, '..', '__init__.py')
        os.environ['TRITON_PTXAS_PATH'] = ptxas_dst

    import triton.backends.nvidia.compiler as nv_compiler
    nv_compiler.get_ptxas_version = lambda arch: '12.0'

    print('Training environment fixes applied.')
else:
    print("USE_PRETRAINED=1: skipping Triton / ptxas environment fixes.")


In [ ]:
# trl installation is handled by the Unsloth offline setup cell below.
if TRAIN_ON_KAGGLE:
    print("Skip standalone trl install/import here; the Unsloth setup cell will install compatible packages.")

In [ ]:
if TRAIN_ON_KAGGLE:
    import glob
    import os
    import subprocess
    import sys

    def recursive_wheels(pattern: str):
        return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))

    packages_dir = "/kaggle/input/datasets/mayukh18/nemotron-packages/packages"
    all_mamba = recursive_wheels("mamba_ssm-*.whl")
    all_causal = recursive_wheels("causal*conv1d*.whl")

    print("Found mamba wheels:", all_mamba)
    print("Found causal-conv1d wheels:", all_causal)

    import torch
    print("Python:", sys.version)
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("Torch CUDA:", torch.version.cuda)

    if not torch.cuda.is_available():
        raise RuntimeError("TRAIN_ON_KAGGLE=1 requires a GPU runtime because Nemotron depends on CUDA wheels.")

    if not os.path.isdir(packages_dir):
        raise FileNotFoundError(f"Offline wheel directory not found: {packages_dir}")

    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "--no-index", "--find-links", packages_dir,
            "unsloth", "trl", "peft", "transformers", "datasets", "accelerate", "bitsandbytes",
        ],
        check=True,
    )

    def pick_last(wheels):
        return wheels[-1] if wheels else None

    causal_wheel = pick_last(all_causal)
    mamba_wheel = pick_last(all_mamba)
    print("Selected causal wheel:", causal_wheel)
    print("Selected mamba wheel:", mamba_wheel)

    if causal_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", causal_wheel], check=True)
    if mamba_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", mamba_wheel], check=True)
    else:
        raise FileNotFoundError("Could not find a compatible mamba_ssm wheel under /kaggle/input.")

    print("Offline package installation finished. Restart the kernel if Kaggle keeps stale imports from earlier runs.")
else:
    print("USE_PRETRAINED=1: skipping datasets / trl / mamba_ssm / unsloth installation.")


In [ ]:
if TRAIN_ON_KAGGLE:
    import torch
    import kagglehub
    from unsloth import FastLanguageModel

    # Patch Unsloth for GRPO compatibility before model load
    try:
        from unsloth import PatchFastRL
        PatchFastRL("GRPO", FastLanguageModel)
        print("PatchFastRL applied for GRPO.")
    except (ImportError, AttributeError):
        print("PatchFastRL not available — proceeding without it.")

    MAX_SEQ_LEN = 8192
    MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
    print(f"Model path: {MODEL_PATH}")

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_PATH,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=False,
        load_in_8bit=False,
        full_finetuning=False,
        trust_remote_code=True,
        unsloth_force_compile=False,
        attn_implementation="eager",
        dtype=torch.bfloat16,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    # GRPO batched generation requires left-padding so all sequences
    # end at the same position before the model generates new tokens.
    tokenizer.padding_side = "left"
    print("Model loaded with Unsloth (GRPO mode).")
else:
    print("USE_PRETRAINED=1: skipping base model and tokenizer loading.")


In [ ]:
if TRAIN_ON_KAGGLE:
    from unsloth import FastLanguageModel

    # ============================================================
    # OPTIMIZED LoRA CONFIG (v9-GRPO) — targeting 92%+ accuracy
    # ============================================================
    # Rank 32 is the competition maximum.
    # RSLoRA uses alpha/sqrt(r) scaling — already regularizes,
    # so dropout=0.0 avoids double-regularization penalty.
    # lm_head EXCLUDED: prevents output distribution drift on \\boxed{}.
    # ============================================================
    LORA_RANK = 32
    LORA_ALPHA = 64          # 2x rank — strong adapter influence
    LORA_DROPOUT = 0.0       # RSLoRA already regularizes; dropout hurts

    target_modules = [
        # Attention projections (core reasoning)
        "q_proj", "k_proj", "v_proj", "o_proj",
        "in_proj", "out_proj",

        # MLP / MoE (feed-forward reasoning capacity)
        "gate_proj", "up_proj", "down_proj",

        # Mamba-specific SSM projections (critical for Nemotron hybrid arch)
        "x_proj", "dt_proj",
        # lm_head intentionally EXCLUDED: training it drifts the output
        # token distribution away from \\boxed{} formatting.
    ]

    print("Creating trainable LoRA wrapper via FastLanguageModel.get_peft_model ...")
    print(f"  Rank={LORA_RANK}, Alpha={LORA_ALPHA}, Dropout={LORA_DROPOUT}")
    print(f"  Target modules: {target_modules}")
    print(f"  RSLoRA=True, lm_head=EXCLUDED")
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=target_modules,
        bias="none",
        use_gradient_checkpointing=True,
        random_state=42,
        use_rslora=True,
    )
    model.print_trainable_parameters()
else:
    print("USE_PRETRAINED=1: skipping trainable LoRA construction.")


## Mode A: Train on Kaggle

In [ ]:
# ============================================================
# MEMORY OPTIMIZATIONS
# FIX: Removed TORCH_CUDA_ALLOC_CONF — conflicted and caused
#      memory fragmentation → CUDA illegal memory access
# ============================================================
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import gc
import math
import re
import subprocess
import time
import zipfile
from pathlib import Path

import pandas as pd
import torch
from datasets import Dataset as HFDataset
from trl import GRPOTrainer, GRPOConfig
from transformers import TrainerCallback

# ============================================================
# GPU METRICS CALLBACK (TensorBoard)
# Logs: GPU util%, memory, temperature, power, throughput
# ============================================================
class GPUMetricsCallback(TrainerCallback):
    def __init__(self, log_every_n_steps=2):
        super().__init__()
        self.log_every_n_steps = log_every_n_steps
        self._last_step_time = None
        self._last_global_step = 0

    def _query_nvidia_smi(self):
        try:
            r = subprocess.run(
                ["nvidia-smi",
                 "--query-gpu=utilization.gpu,memory.used,memory.total,"
                 "temperature.gpu,power.draw",
                 "--format=csv,noheader,nounits"],
                capture_output=True, text=True, timeout=5)
            if r.returncode != 0:
                return None
            p = [x.strip() for x in r.stdout.strip().split("\n")[0].split(",")]
            return {
                "gpu/utilization_percent": float(p[0]),
                "gpu/memory_used_gb":      float(p[1]) / 1024.0,
                "gpu/temperature_celsius": float(p[3]),
                "gpu/power_watts":         float(p[4]) if p[4] != "[N/A]" else 0.0,
            }
        except Exception:
            return None

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None or state.global_step % self.log_every_n_steps != 0:
            return
        smi = self._query_nvidia_smi()
        if smi:
            logs.update(smi)
        if torch.cuda.is_available():
            logs["gpu/memory_allocated_gb"]     = torch.cuda.memory_allocated()     / (1024**3)
            logs["gpu/memory_reserved_gb"]      = torch.cuda.memory_reserved()      / (1024**3)
            logs["gpu/max_memory_allocated_gb"] = torch.cuda.max_memory_allocated() / (1024**3)
        now = time.time()
        if self._last_step_time is not None:
            elapsed = now - self._last_step_time
            steps   = state.global_step - self._last_global_step
            if elapsed > 0 and steps > 0:
                sps = steps / elapsed
                logs["throughput/steps_per_sec"]   = sps
                logs["throughput/samples_per_sec"] = sps * args.per_device_train_batch_size
        self._last_step_time    = now
        self._last_global_step  = state.global_step

    def on_train_begin(self, args, state, control, **kwargs):
        self._last_step_time   = time.time()
        self._last_global_step = state.global_step
        print("[TensorBoard] GPU metrics logging enabled.")
        smi = self._query_nvidia_smi()
        if smi:
            print(f"  GPU: {smi['gpu/utilization_percent']:.0f}% util | "
                  f"{smi['gpu/memory_used_gb']:.1f} GB | "
                  f"{smi['gpu/temperature_celsius']:.0f}C")

    def on_train_end(self, args, state, control, **kwargs):
        smi = self._query_nvidia_smi()
        if smi:
            peak = torch.cuda.max_memory_allocated() / (1024**3)
            print(f"[TensorBoard] Done. Peak mem: {peak:.1f} GB | "
                  f"Temp: {smi['gpu/temperature_celsius']:.0f}C")

print("Imports done. GRPOTrainer, GRPOConfig, GPUMetricsCallback ready.")


## Rewards

In [ ]:
if TRAIN_ON_KAGGLE:
    import re

    # ============================================================
    # REWARD FUNCTIONS FOR GRPO — Math Competition
    # ============================================================
    # Signal priority:
    #   1. Correctness  (2.0 max) — primary RL signal; binary match
    #   2. Format       (1.0 max) — ensures <think>+\boxed{} structure
    #   3. Length       (0.3 max) — encourages substantive reasoning
    #   4. Repetition   (0.1 / -0.3) — guards against degenerate loops
    # Combined max ≈ 3.4 per completion.
    # ============================================================

    def _extract_boxed(text: str) -> str | None:
        """Extract content from last \\boxed{} with nested-brace support."""
        pos = text.rfind(r"\boxed{")
        if pos == -1:
            return None
        start = pos + len(r"\boxed{")
        depth, i = 1, start
        while i < len(text) and depth > 0:
            if text[i] == "{":
                depth += 1
            elif text[i] == "}":
                depth -= 1
            i += 1
        return text[start : i - 1].strip() if depth == 0 else None

    def _normalize(ans: str) -> str:
        """Normalize for comparison: strip LaTeX noise, unify numeric form."""
        if not ans:
            return ""
        ans = ans.strip()
        for tok in (r"\\,", r"\,", r"\\!", r"\!", r"\\ ", r"\ ", r"\text{", "}"):
            ans = ans.replace(tok, "")
        try:
            val = float(ans.replace(",", ""))
            if val == int(val) and abs(val) < 1e15:
                return str(int(val))
            return f"{val:.8g}"
        except (ValueError, OverflowError):
            pass
        return ans.lower().replace(" ", "")

    def _is_correct(pred: str | None, gold: str) -> bool:
        if pred is None:
            return False
        p, g = _normalize(pred), _normalize(str(gold))
        if p == g:
            return True
        try:
            pv, gv = float(p.replace(",", "")), float(g.replace(",", ""))
            return abs(pv - gv) < 1e-6 or (gv != 0 and abs(pv - gv) / abs(gv) < 1e-5)
        except (ValueError, OverflowError):
            pass
        return False

    # ------------------------------------------------------------------
    # Component reward functions
    # Signature: (completions: list[str], **kwargs) -> list[float]
    # kwargs contains all extra dataset columns (e.g. kwargs["answer"]).
    # ------------------------------------------------------------------

    def reward_correctness(completions: list[str], answer: list[str], **kwargs) -> list[float]:
        """2.0 if boxed answer matches gold, else 0.0."""
        return [
            2.0 if _is_correct(_extract_boxed(c), str(a)) else 0.0
            for c, a in zip(completions, answer)
        ]

    def reward_format(completions: list[str], **kwargs) -> list[float]:
        """Up to 1.0 for correct <think>…</think> + \\boxed{} structure."""
        rewards = []
        for c in completions:
            score = 0.0
            has_open  = "<think>" in c
            has_close = "</think>" in c
            has_boxed = r"\boxed{" in c
            extracted = _extract_boxed(c)

            if has_open and has_close:
                score += 0.3
                think_end = c.rfind("</think>")
                boxed_pos = c.rfind(r"\boxed{")
                if boxed_pos > think_end > 0:   # think block precedes answer
                    score += 0.1
            if has_boxed:
                score += 0.4
            if extracted is not None and len(extracted) > 0:
                score += 0.2

            rewards.append(min(score, 1.0))
        return rewards

    def reward_reasoning_length(completions: list[str], **kwargs) -> list[float]:
        """
        Encourage substantive but focused reasoning. Max 0.3.
        Ramp 0→0.3 over 0–2000 chars, plateau 2000–5500, decay to 0 at 8000.
        """
        RAMP_TO    = 2000
        PLATEAU_TO = 5500
        DECAY_TO   = 8000
        rewards = []
        for c in completions:
            m = re.search(r"<think>(.*?)</think>", c, re.DOTALL)
            n = len(m.group(1)) if m else 0
            if n <= RAMP_TO:
                score = 0.3 * n / RAMP_TO
            elif n <= PLATEAU_TO:
                score = 0.3
            elif n <= DECAY_TO:
                score = 0.3 * (DECAY_TO - n) / (DECAY_TO - PLATEAU_TO)
            else:
                score = 0.0
            rewards.append(score)
        return rewards

    def reward_no_repetition(completions: list[str], **kwargs) -> list[float]:
        """6-gram diversity in last 200 words. +0.1 diverse, -0.3 heavy loop."""
        rewards = []
        for c in completions:
            words = c.split()
            if len(words) < 30:
                rewards.append(0.0)
                continue
            tail   = words[-min(len(words), 200):]
            ngrams = [tuple(tail[i : i + 6]) for i in range(len(tail) - 5)]
            if not ngrams:
                rewards.append(0.0)
                continue
            div = len(set(ngrams)) / len(ngrams)
            if div >= 0.85:
                rewards.append(0.1)
            elif div >= 0.65:
                rewards.append(0.0)
            elif div >= 0.45:
                rewards.append(-0.15)
            else:
                rewards.append(-0.3)
        return rewards

    def reward_combined(completions: list[str], answer: list[str], **kwargs) -> list[float]:
        """Weighted sum of all reward signals (used as GRPO reward_func)."""
        cor = reward_correctness(completions, answer, **kwargs)
        fmt = reward_format(completions, **kwargs)
        lng = reward_reasoning_length(completions, **kwargs)
        rep = reward_no_repetition(completions, **kwargs)
        return [c + f + l + r for c, f, l, r in zip(cor, fmt, lng, rep)]

    REWARD_FUNCS = [reward_combined]

    print("Reward functions defined:")
    print("  reward_correctness     2.0 max  (exact match + numeric tolerance)")
    print("  reward_format          1.0 max  (<think> tags + \\boxed{} present & ordered)")
    print("  reward_reasoning_length 0.3 max  (ramp 0→2k, plateau→5.5k, decay→8k)")
    print("  reward_no_repetition   0.1 / -0.3  (6-gram diversity guard)")
    print("  reward_combined        used as GRPO reward_func (max ≈ 3.7)")
else:
    print("USE_PRETRAINED=1: skipping reward function definitions.")


## GRPO-Trainer

In [ ]:
if TRAIN_ON_KAGGLE:
    # ============================================================
    # GRPO DATASET LOADING
    # ============================================================
    SEED = 42
    PROMPT_SUFFIX  = "\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`"
    SYSTEM_PROMPT  = (
        "You are an expert mathematics assistant. "
        "Think carefully step by step inside <think>...</think> tags, "
        "then give your final answer inside \\boxed{}."
    )
    DATASET_PATH = "/kaggle/input/datasets/dgxchen/nemotron-cot-tong/problem_ids_matched.csv"

    df = pd.read_csv(DATASET_PATH)
    df = df.dropna(subset=["answer", "prompt"]).reset_index(drop=True)
    df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
    print(f"Dataset: {len(df)} rows after filtering")

    # GRPO needs prompt-only inputs; the model generates its own CoT.
    # Ground-truth answer is passed as a dataset column → reward functions.
    grpo_records = []
    for _, row in df.iterrows():
        grpo_records.append({
            "prompt": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": str(row["prompt"]) + PROMPT_SUFFIX},
            ],
            "answer": str(row["answer"]),
        })

    grpo_dataset = HFDataset.from_list(grpo_records)
    print(f"GRPO dataset: {len(grpo_dataset)} samples")

    # ============================================================
    # GRPO CONFIG — v9, tuned for 30B Nemotron on Kaggle GPU
    # ============================================================
    TB_LOG_DIR = "/kaggle/working/tb_logs"

    grpo_config = GRPOConfig(
        # -- Core GRPO --
        num_generations=4,          # completions per prompt; 4 balances signal vs VRAM
        max_new_tokens=3072,        # enough room for full CoT + boxed answer
        temperature=0.7,            # exploration-exploitation balance
        top_p=0.9,
        beta=0.02,                  # KL-penalty weight; 0.02 = light regularization

        # -- Optimization --
        learning_rate=5e-6,         # RL needs much smaller LR than SFT
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        num_train_epochs=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,   # effective batch = 8 prompts × 4 gen = 32 completions
        optim="adamw_8bit",
        adam_beta1=0.9,
        adam_beta2=0.999,
        adam_epsilon=1e-8,
        weight_decay=0.01,
        max_grad_norm=0.3,          # tighter clip for RL stability

        # -- Memory --
        gradient_checkpointing=True,
        bf16=True,

        # -- Logging --
        output_dir="/kaggle/working/grpo_output",
        logging_steps=1,
        logging_dir=TB_LOG_DIR,
        report_to="tensorboard",
        save_strategy="no",

        # -- Misc --
        seed=SEED,
        remove_unused_columns=False,
        dataloader_num_workers=2,
    )

    print("\n" + "=" * 60)
    print("  GRPO CONFIG v9")
    print("=" * 60)
    print(f"  LR:              {grpo_config.learning_rate}")
    print(f"  num_generations: {grpo_config.num_generations}")
    print(f"  max_new_tokens:  {grpo_config.max_new_tokens}")
    print(f"  temperature:     {grpo_config.temperature}")
    print(f"  beta (KL):       {grpo_config.beta}")
    bs = grpo_config.per_device_train_batch_size
    ga = grpo_config.gradient_accumulation_steps
    print(f"  Batch:           {bs} prompt × {ga} accum × {grpo_config.num_generations} gen")
    print(f"  Epochs:          {grpo_config.num_train_epochs}")
    print(f"  TensorBoard:     {TB_LOG_DIR}")
    print("=" * 60 + "\n")

    # ============================================================
    # GRPO TRAINER
    # ============================================================
    torch.cuda.empty_cache()
    gc.collect()

    trainer = GRPOTrainer(
        model=model,
        processing_class=tokenizer,
        reward_funcs=REWARD_FUNCS,
        args=grpo_config,
        train_dataset=grpo_dataset,
        callbacks=[GPUMetricsCallback(log_every_n_steps=1)],
    )

    print("Starting GRPO training v9...")
    t0 = time.time()
    trainer.train()
    elapsed = time.time() - t0
    print(f"GRPO training done in {elapsed / 60:.1f} min")

    ADAPTER_DIR = "/kaggle/working/sft_adapter"
    model.save_pretrained(ADAPTER_DIR)
    tokenizer.save_pretrained(ADAPTER_DIR)
    print(f"Adapter saved to {ADAPTER_DIR}")
else:
    print("USE_PRETRAINED=1: skipping GRPO training.")


## Package TensorBoard Logs for Download

In [ ]:
if TRAIN_ON_KAGGLE:
    import os, zipfile
    from pathlib import Path

    TB_LOG_DIR = "/kaggle/working/tb_logs"
    ZIP_OUTPUT = "/kaggle/working/tensorboard_logs.zip"

    log_path = Path(TB_LOG_DIR)
    if log_path.exists():
        files = [f for f in log_path.rglob("*") if f.is_file()]
        events = list(log_path.rglob("events.out.tfevents.*"))
        print(f"\n{'='*60}")
        print(f"  PACKAGING TENSORBOARD LOGS ({len(events)} event files, {len(files)} total)")
        print(f"{'='*60}")
        with zipfile.ZipFile(ZIP_OUTPUT, "w", zipfile.ZIP_DEFLATED) as zf:
            for fp in files:
                zf.write(fp, fp.relative_to(log_path.parent))
        sz = os.path.getsize(ZIP_OUTPUT) / (1024*1024)
        print(f"  => {ZIP_OUTPUT} ({sz:.2f} MB)")
        print(f"\n  TO VIEW LOCALLY:")
        print(f"  unzip tensorboard_logs.zip")
        print(f"  pip install tensorboard")
        print(f"  tensorboard --logdir=tb_logs/ --port=6006")
        print(f"  # Open http://localhost:6006")
        print(f"{'='*60}\n")
    else:
        print(f"[WARN] No TB logs at {TB_LOG_DIR}")


## Mode B: Load Pre-trained LoRA（Temporarily unavailable）

In [ ]:
if USE_PRETRAINED:
    import os

    SRC_ADAPTER_DIR = PRETRAINED_ADAPTER_DATASET_PATH
    required_files = ["adapter_config.json", "adapter_model.safetensors"]

    print("Using pre-trained adapter from:", SRC_ADAPTER_DIR)
    for fname in required_files:
        fpath = os.path.join(SRC_ADAPTER_DIR, fname)
        if not os.path.exists(fpath):
            raise FileNotFoundError(f"Missing required adapter file: {fpath}")
        print(f"  {fname}: {os.path.getsize(fpath)/1024/1024:.1f} MB")
else:
    print("TRAIN_ON_KAGGLE=1: pretrained adapter path check skipped.")


## Create submission.zip

In [ ]:
import json, os, shutil, zipfile

OUTPUT_DIR = "/kaggle/working"
SUBMISSION_ADAPTER_DIR = os.path.join(OUTPUT_DIR, "submission_adapter")
os.makedirs(SUBMISSION_ADAPTER_DIR, exist_ok=True)

required_files = ["adapter_config.json", "adapter_model.safetensors"]

if TRAIN_ON_KAGGLE:
    src_adapter_dir = "/kaggle/working/sft_adapter"
    print("Packaging freshly trained adapter from:", src_adapter_dir)
else:
    src_adapter_dir = PRETRAINED_ADAPTER_DATASET_PATH
    print("Packaging pre-trained adapter directly from:", src_adapter_dir)

for fname in required_files:
    src = os.path.join(src_adapter_dir, fname)
    dst = os.path.join(SUBMISSION_ADAPTER_DIR, fname)
    if not os.path.exists(src):
        raise FileNotFoundError(f"Missing required adapter file: {src}")
    shutil.copy2(src, dst)
    print(f"Copied {fname} ({os.path.getsize(dst)/1024/1024:.1f} MB)")

config_path = os.path.join(SUBMISSION_ADAPTER_DIR, "adapter_config.json")
with open(config_path, "r") as f:
    cfg = json.load(f)

cfg["base_model_name_or_path"] = BASE_MODEL_NAME
cfg["inference_mode"] = True
cfg["lora_dropout"] = 0.0

with open(config_path, "w") as f:
    json.dump(cfg, f, indent=2)

zip_path = os.path.join(OUTPUT_DIR, "submission.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in required_files:
        fpath = os.path.join(SUBMISSION_ADAPTER_DIR, fname)
        zf.write(fpath, fname)
        print(f"  Added {fname}")

zip_sz = os.path.getsize(zip_path) / 1024 / 1024
print(f"\nsubmission.zip: {zip_sz:.1f} MB")
print("Done! Ready to submit.")
